# 은행 고객 이탈 예측 v2 — 재현 가능한 고객 선별

수업 프로젝트를 기반으로 평가 절차·전처리 일관성·연락 예산 관점의 평가를 개선했습니다. 전체 모델링은 `churn/` 모듈에 있으며 이 노트북은 같은 코드를 호출합니다. 예전 실행 출력은 제거했습니다.

원본 데이터는 별도로 준비합니다. 합성 데이터로 실행하면 실제 금융 성능으로 해석하지 않습니다.

## 1. 실행 환경

프로젝트 루트에서 `pip install -r requirements.txt`를 먼저 실행하고 해당 Python 환경을 노트북 커널로 선택하세요. 신규 다운로드·유료 API 호출은 코드에 없습니다.

In [ ]:
import os
from pathlib import Path
import json
import joblib
import pandas as pd
from churn.train import run, score

DATA = Path(os.getenv("CHURN_DATA", "data/bank_churn_train.csv"))
OUTPUT = Path(os.getenv("CHURN_RUN", "artifacts/notebook-v2"))
if not DATA.is_file():
    raise FileNotFoundError("데이터를 별도로 준비하거나 CHURN_DATA를 지정하세요. 합성 점검은 python -m churn.demo --output data/demo.csv")


## 2. 데이터 점검

고객 ID의 중복을 거부하며 ID·성씨·정답·허용 목록 밖 컬럼은 모델 입력에서 제외합니다. 개인별 원본 행은 출력하지 않습니다.

In [ ]:
data = pd.read_csv(DATA)
print("rows:", len(data))
print("target counts:", data.Exited.value_counts().to_dict())


## 3. 모델 선택과 최종 평가

Train 60%에서 후보 5개를 학습합니다. Validation 20%의 상위 10% Precision → AP → ROC-AUC 순서로 모델을 선택하고 같은 Validation에서 F2 임계값을 정합니다. 선택 모델을 Train+Validation으로 재학습한 뒤 Test 20%에서 한 번만 예측합니다. 기존 결과 폴더는 덮어쓰지 않으므로 재실행할 때 CHURN_RUN을 새 경로로 지정하세요.

In [ ]:
import hashlib
tracking = os.getenv("CHURN_TRACKING_DIR")  # 선택: requirements-tracking.txt 설치 후 지정
report = run(data, OUTPUT, seed=42, fraction=.1,
             dataset_sha256=hashlib.sha256(DATA.read_bytes()).hexdigest(),
             tracking_dir=Path(tracking) if tracking else None)
print("selected:", report["selected_model"])
print("split:", report["split"])
pd.DataFrame(report["validation_candidates"])[["model", "roc_auc", "average_precision", "precision_at_budget", "recall_at_budget", "lift_at_budget"]]


## 4. 최종 Test 결과

임계값 기반 risk_flag와 정확히 상위 10%만 선택하는 contact_selected는 다른 정책입니다. 아래 값으로 모델·임계값·예산을 다시 조정하지 않습니다. AP는 average precision이며 사다리꼴 적분 PR-AUC와 동일한 지표가 아닙니다.

In [ ]:
pd.Series(report["final_test"])


In [ ]:
pd.DataFrame(report["test_contact_budget_curve"])[["contact_fraction", "contacted", "captured_churners", "precision_at_budget", "recall_at_budget", "lift_at_budget"]]


## 5. 저장된 모델로 신규 고객 처리

원시 컬럼을 그대로 입력합니다. 파생변수·결측값·인코딩·스케일링은 저장된 Pipeline이 동일하게 수행합니다. 본인이 생성한 신뢰 가능한 joblib 파일만 불러오세요. 개별 점수는 Git에서 제외된 artifacts 아래에만 저장합니다.

In [ ]:
NEW_DATA = Path(os.getenv("CHURN_NEW", "data/bank_churn_new.csv"))
if NEW_DATA.is_file():
    bundle = joblib.load(OUTPUT / "model.joblib")
    scored = score(pd.read_csv(NEW_DATA), bundle)
    with (OUTPUT / "new_customer_scores.csv").open("x") as handle:
        scored.to_csv(handle, index=False)
    print("scored:", len(scored), "contact_selected:", int(scored.contact_selected.sum()))
else:
    print("신규 고객 파일이 없어 scoring을 생략했습니다.")


## 해석과 한계

이 교육 데이터는 이미 예전 노트북에서 분석한 자료여서 완전히 새로운 외부 홀드아웃이 아닙니다. 같은 기준일의 고객 스냅샷이므로 미래 시점 검증도 아닙니다. 연락 대상 cohort가 작아 선택 결과가 변동될 수 있습니다. 이탈 가능성이 높다는 것이 연락으로 이탈을 방지할 수 있다는 뜻은 아니며 uplift·순이익을 증명하지 않습니다. `reports/churn-v2.json`은 별도 CLI 실행의 집계 결과이며 이 노트북 실행 출력이 아닙니다.